# HealthConnect Clinic
## Week 4 Initial Data Analytics Analysis

**Track:** Data Analytics
**Tools:** Python
**Project Focus:** Appointment Attendance and No-Show Patterns
---
### Objective
To understand the HealthConnect appointment dataset and identify how it can be used to investigate appointment attendance and no-show patterns.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("HealthConnect_Appointment_Data.csv")

df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [3]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

Number of rows: 5000
Number of columns: 18


In [4]:
print("Column names:")
for column in df.columns:
    print(column)

Column names:
appointment_id
patient_id
gender
age
age_group
appointment_type
booking_date
appointment_date
appointment_day
appointment_time
booking_lead_days
previous_appointments
previous_no_shows
reminder_sent
reminder_channel
distance_to_clinic_km
waiting_time_minutes
appointment_outcome


In [5]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

missing[missing["Missing Values"] > 0]

,Missing Values,Missing Percentage
reminder_channel,1366,27.32
distance_to_clinic_km,90,1.80
waiting_time_minutes,60,1.20


## 2. Initial Data Quality Assessment
### 2.1 Missing Values
The dataset was assessed for missing values across all variables.
The purpose of this check was to identify variables containing incomplete records that may require further investigation before subsequent analysis.
The results of the missing-value assessment are shown above.

### Initial Observation
Missing values were identified in the variables shown in the table above. These values will be investigated further using the Data Dictionary to determine whether they represent genuine missing information or expected values within the dataset structure.
No values will be removed or replaced at this stage.

In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [7]:
print("Duplicate appointment IDs:",
      df["appointment_id"].duplicated().sum())

Duplicate appointment IDs: 0


### 2.2 Duplicate Records
The dataset was checked for duplicate rows and duplicate appointment IDs.
The appointment_id field was specifically checked because it is used to uniquely identify an appointment.
The results of both duplicate checks are shown above.

### Initial Observation
The duplicate checks will be used to determine whether the dataset contains repeated appointment records that could affect subsequent analysis.

In [8]:
df["appointment_outcome"].value_counts(dropna=False)

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

### 2.3 Appointment Outcome Validation
The appointment_outcome variable was reviewed because it represents the final status of each appointment and is the primary outcome variable for the attendance and no-show analysis.
The frequency of each outcome category was examined to verify the values present in the dataset.

### Initial Observation
The observed appointment outcome categories will be compared with the categories specified in the Data Dictionary. This variable will form the basis of the subsequent appointment attendance and no-show analysis.

In [9]:
invalid_history = df[
    df["previous_no_shows"] > df["previous_appointments"]
]

print("Records where previous no-shows exceed previous appointments:",
      len(invalid_history))

Records where previous no-shows exceed previous appointments: 0


In [10]:
invalid_history[
    ["patient_id",
     "previous_appointments",
     "previous_no_shows"]
].head()

,patient_id,previous_appointments,previous_no_shows


### 2.4 Previous Appointment History Consistency
The relationship between previous_appointments and previous_no_shows was assessed.
A patient's number of previous no-shows should not exceed the total number of previous appointments.
This check was conducted to identify potentially inconsistent historical appointment records.

### Initial Observation
The number of records identified by the validation check is shown above. Any records that fail this condition will require further investigation before subsequent analysis.

In [11]:
pd.crosstab(
    df["reminder_sent"],
    df["reminder_channel"],
    dropna=False
)

reminder_channel,Email,SMS,WhatsApp,NaN
reminder_sent,,,,
No,0,0,0,1366
Yes,533,2000,1101,0


In [12]:
df.groupby(
    ["reminder_sent", "reminder_channel"],
    dropna=False
).size().reset_index(name="Count")

,reminder_sent,reminder_channel,Count
0,No,NaN,1366
1,Yes,Email,533
2,Yes,SMS,2000
3,Yes,WhatsApp,1101


In [13]:
print("Reminder sent = No but channel provided:")

display(
    df[
        (df["reminder_sent"] == "No") &
        (df["reminder_channel"].notna())
    ]
)

Reminder sent = No but channel provided:


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome


In [14]:
print("Reminder sent = Yes but channel missing:")

display(
    df[
        (df["reminder_sent"] == "Yes") &
        (df["reminder_channel"].isna())
    ]
)

Reminder sent = Yes but channel missing:


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome


### 2.5 Reminder Data Consistency
The relationship between reminder_sent and reminder_channel was examined to determine whether the reminder information is internally consistent.
This check is important because the Data Dictionary defines the expected relationship between whether a reminder was sent and the corresponding reminder channel.

### Initial Observation
The results of the reminder consistency checks will be considered when preparing the dataset for subsequent analysis. No values have been modified at this stage

##2.6 Date Validation
The booking and appointment date variables were reviewed to assess whether they contain valid date values and are suitable for subsequent analysis.
The date fields were converted to datetime format and checked for invalid or missing values

In [15]:
df["booking_date"] = pd.to_datetime(
    df["booking_date"],
    errors="coerce"
)

df["appointment_date"] = pd.to_datetime(
    df["appointment_date"],
    errors="coerce"
)

print("Invalid or missing booking dates:",
      df["booking_date"].isna().sum())

print("Invalid or missing appointment dates:",
      df["appointment_date"].isna().sum())

Invalid or missing booking dates: 0
Invalid or missing appointment dates: 0


In [16]:
invalid_dates = df[
    df["appointment_date"] < df["booking_date"]
]

print(
    "Appointments occurring before booking date:",
    len(invalid_dates)
)

Appointments occurring before booking date: 0


In [17]:
invalid_dates[
    ["booking_date", "appointment_date"]
].head()

,booking_date,appointment_date


### Appointment and Booking Date Consistency
A consistency check was performed to identify records where the appointment date occurred before the booking date.
This check helps identify potentially inconsistent scheduling records before the dataset is used for subsequent analysis

### 2.7 Booking Lead Days Validation
The booking_lead_days variable was validated by comparing the recorded value with the number of days between the booking date and appointment date.
This assessment was conducted to evaluate the internal consistency of the booking information

In [18]:
df["calculated_lead_days"] = (
    df["appointment_date"] - df["booking_date"]
).dt.days

df[
    [
        "booking_date",
        "appointment_date",
        "booking_lead_days",
        "calculated_lead_days"
    ]
].head()

,booking_date,appointment_date,booking_lead_days,calculated_lead_days
0,2025-02-06,2025-02-18,12,12
1,2026-02-25,2026-02-27,2,2
2,2025-11-16,2025-12-24,38,38
3,2025-07-18,2025-08-28,41,41
4,2025-07-09,2025-08-25,47,47


In [19]:
lead_day_mismatch = df[
    df["booking_lead_days"] != df["calculated_lead_days"]
]

print(
    "Booking lead-day mismatches:",
    len(lead_day_mismatch)
)

Booking lead-day mismatches: 0


In [20]:
lead_day_mismatch[
    [
        "booking_date",
        "appointment_date",
        "booking_lead_days",
        "calculated_lead_days"
    ]
].head()

,booking_date,appointment_date,booking_lead_days,calculated_lead_days


### Initial Observation
The recorded booking_lead_days values were compared with values calculated from the booking and appointment dates.
The number of mismatched records identified by the validation check is reported above. Any discrepancies will be considered during subsequent data preparation

### 2.8 Numerical Variable Checks
Descriptive statistics were generated for the main numerical variables to assess their ranges and identify potentially unusual values.
The variables assessed include age, booking lead days, previous appointments, previous no-shows, distance to the clinic and waiting time.

In [21]:
numerical_columns = [
    "age",
    "booking_lead_days",
    "previous_appointments",
    "previous_no_shows",
    "distance_to_clinic_km",
    "waiting_time_minutes"
]

df[numerical_columns].describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


In [22]:
for column in numerical_columns:
    negative_values = (df[column] < 0).sum()
    print(f"{column}: {negative_values} negative values")

age: 0 negative values
booking_lead_days: 0 negative values
previous_appointments: 0 negative values
previous_no_shows: 0 negative values
distance_to_clinic_km: 0 negative values
waiting_time_minutes: 0 negative values


### 2.9 Data Quality Assessment Summary
The initial data-quality assessment examined missing values, duplicate records, appointment ID uniqueness, appointment outcome categories, historical appointment consistency, reminder data consistency, date validity, booking lead-time consistency and numerical variable ranges.
The assessment provides an initial understanding of the strengths and potential limitations of the dataset.
No records were removed and no missing or unusual values were replaced during this initial assessment. Any issues identified will be considered during the data-preparation stage of the project

# 3. Important Variables Relevant to Appointment Attendance and No-Shows

The variables in the HealthConnect appointment dataset were reviewed to
identify those most relevant to investigating appointment attendance and
no-show patterns.

The variables were grouped according to their role in the analysis,
including appointment outcomes, patient history, reminders, appointment
characteristics, booking information, operational factors and patient
demographics.

## 3.1 Appointment Outcome
### appointment_outcome
**Role:** Primary outcome variable
**Relevance**
appointment_outcome is the most important variable for this project because it records the final status of an appointment. It allows the analysis to distinguish between attended appointments, no-shows and cancellations.
This variable will therefore be used as the main outcome against which other variables can be examined in subsequent analysis

## 3.2 Previous Appointment Behaviour
## previous_appointments
**Role:** Historical patient behaviour
**Relevance:**
This variable indicates the patient's previous appointment history and may help determine whether the amount of previous appointment activity is associated with future attendance patterns.
## previous_no_shows ##
**Role:** Historical no-show behaviour
**Relevance:**
This variable is particularly important because previous missed appointments may be associated with future no-show behaviour. It can therefore be used to investigate whether previous attendance behaviour is relevant to subsequent appointment outcomes.

## 3.3 Reminder Variables
### reminder_sent
**Role:** Reminder activity
**Relevance:**
This variable identifies whether an appointment reminder was sent. It is important for investigating whether reminder activity is associated with appointment attendance and no-shows.
## reminder_channel
**Role:** Reminder delivery method
**Relevance:**
This variable identifies the channel through which a reminder was delivered. It may allow subsequent analysis to compare appointment outcomes across different reminder channels 

## 3.4 Appointment Scheduling Variables
### appointment_type
**Role:** Appointment characteristics
**Relevance:**
Appointment type may be associated with different attendance patterns. Comparing outcomes across appointment types can help determine whether certain types of appointments experience higher levels of no-shows.
appointment_day
Role: Day of appointment
Relevance:
The day of the week may be associated with differences in appointment attendance. This variable can therefore be used to investigate whether no-show patterns vary across appointment days.
### appointment_time
**Role:** Time period
**Relevance:**
Appointment time may influence attendance patterns. Comparing morning, afternoon and evening appointments may help identify differences in attendance and no-show behaviour

## 3.5 Booking Information
### booking_lead_days
**Role:** Time between booking and appointment
**Relevance:**
Booking lead time is important because the amount of time between booking an appointment and the scheduled appointment may be associated with the likelihood of attendance or a no-show.
This variable will therefore be investigated in relation to appointment_outcome

## 3.6 Operational and Accessibility Variables
### distance_to_clinic_km
**Role:** Distance/accessibility
**Relevance:**
Distance to the clinic may provide an indication of accessibility. The variable can be investigated to determine whether appointment outcomes vary according to the approximate distance patients are from the clinic.
### waiting_time_minutes
**Role:** Operational factor
**Relevance:**
Waiting time may be relevant to the patient experience and could be investigated in relation to appointment outcomes

## 3.7 Demographic Variables
### age and age_group
**Role:** Patient demographics
**Relevance:**
Age and age group can be used to describe appointment outcomes across different patient age categories and investigate whether attendance patterns vary across age groups.
### gender
**Role:** Patient demographic
**Relevance:**
Gender can be used for descriptive comparison of appointment outcomes across the recorded gender categories.

## 3.8 Important Variable Summary
|Variable Category Relevance to Analysis|
|---|---|---|
|'appointment_outcome'| Outcome Main variable used to identify attendance, no-shows and cancellations|
| 'previous_appointment'|s Patient history Provides information about previous appointment activity|
| 'previous_no_shows'|Patient history Helps investigate whether previous no-show behaviour is associated with future no-shows|
| 'reminder_sent'| Reminder Allows investigation of reminder activity and appointment outcomes|
| 'reminder_channel'| Reminder Allows comparison of outcomes across reminder channels|
| 'appointment_type| Appointment Allows comparison of attendance patterns across appointment types|
| 'appointment_day| Scheduling Allows investigation of day-of-week attendance patterns|
| 'appointment_time| Scheduling Allows investigation of attendance patterns across time periods|
| 'booking_lead_days |Booking Allows investigation of the relationship between booking lead time and attendance|
| 'distance_to_clinic_km |Accessibility Allows investigation of whether distance may be associated with appointment outcomes|
| 'waiting_time_minutes| Operations Allows investigation of whether waiting time is associated with appointment outcomes|
| 'age| Demographic Allows descriptive comparison of outcomes across ages|
|'age_group' |Demographic Allows descriptive comparison across age bands|
| 'gender'| Demographic Allows descriptive comparison of appointment outcomes across recorded gender categories|

## 3.9 Variable Selection Summary
The review identified appointment_outcome as the primary outcome variable for the project.
The variables considered most directly relevant to investigating attendance and no-show patterns are previous_no_shows, previous_appointments, reminder_sent, reminder_channel, appointment_type, appointment_day, appointment_time, booking_lead_days, distance_to_clinic_km and waiting_time_minutes.
Demographic variables including age, age_group and gender will support descriptive comparisons.
These variables will form the foundation for the business questions and subsequent exploratory analysis

## 4. Business Questions

The business questions below were developed based on the project objective
of understanding appointment attendance and no-show patterns at HealthConnect
Clinic.

The questions focus on identifying patterns in appointment outcomes and
examining factors that may be associated with attendance, no-shows and
cancellations.

These questions will guide the subsequent exploratory analysis and KPI
development

## BQ1. What proportion of scheduled appointments are attended, missed
or cancelled?
Relevant variable: appointment_outcome
Purpose:
This question establishes the overall appointment outcome pattern and provides a baseline understanding of attendance, no-shows and cancellations within the dataset.

## BQ2. Is reminder activity associated with appointment attendance and
no-show patterns?
Relevant variables:
reminder_sent
reminder_channel
appointment_outcome
Purpose:
This question investigates whether appointment outcomes differ depending on whether a reminder was sent and, where applicable, which reminder channel was used

## BQ3. Are patients with a history of previous no-shows more likely to
miss future appointments?
Relevant variables:
previous_appointments
previous_no_shows
appointment_outcome
Purpose:
This question investigates whether previous appointment behaviour is associated with the outcome of subsequent appointments.
The analysis may help identify whether historical no-show behaviour is an important factor to consider when understanding appointment attendance.

## BQ4. Do appointment attendance and no-show patterns vary by appointment
type, day of the week or time of day?
Relevant variables:
appointment_type
appointment_day
appointment_time
appointment_outcome
Purpose:
This question investigates whether certain appointment types or scheduling periods are associated with different attendance and no-show patterns.

## BQ5. Is the amount of time between booking and the appointment associated
with appointment no-shows?
Relevant variables:
booking_date
appointment_date
booking_lead_days
appointment_outcome
Purpose:
This question investigates whether booking lead time is associated with appointment attendance and no-show patterns.

## BQ6. Are distance to the clinic and waiting time associated with
appointment attendance and no-shows?
Relevant variables:
distance_to_clinic_km
waiting_time_minutes
appointment_outcome
Purpose:
This question investigates whether accessibility and operational factors are associated with differences in appointment outcomes.

## BQ7. Do appointment attendance and no-show patterns differ across
patient demographic groups?
Relevant variables:
age
age_group
gender
appointment_outcome
Purpose:
This question supports descriptive analysis of appointment outcomes across the demographic groups represented in the dataset.

## 4.1 Business Question Summary
|ID Business Question Key Variables|
|BQ1 |What proportion of scheduled appointments are attended, missed or cancelled? appointment_outcome|
|BQ2| Is reminder activity associated with appointment attendance and no-show patterns? reminder_sent, reminder_channel, appointment_outcome|
|BQ3 |Are patients with a history of previous no-shows more likely to miss future appointments? previous_appointments, previous_no_shows, appointment_outcome|
|BQ4 |Do attendance and no-show patterns vary by appointment type, day or time? appointment_type, appointment_day, appointment_time, appointment_outcome|
|BQ5 |Is booking lead time associated with appointment no-shows? booking_lead_days, appointment_outcome
|BQ6 |Are distance to the clinic and waiting time associated with attendance and no-shows? distance_to_clinic_km, waiting_time_minutes, appointment_outcome|
|BQ7| Do attendance and no-show patterns differ across demographic groups? age, age_group, gender, appointment_outcome|

## 5. Proposed KPIs

The following potential KPIs were identified based on the business questions
developed for the HealthConnect appointment analysis.

The KPIs are intended to provide measurable indicators for evaluating
appointment attendance, no-shows, cancellations and reminder activity.

At this stage, the KPIs are only being identified and justified. They will
be calculated and analysed during subsequent stages of the project.

## KPI 1 — No-Show Rate
**Linked Business Question:** BQ1 — What proportion of scheduled appointments are attended, missed or cancelled?
**Definition:**
The percentage of scheduled appointments that result in a no-show.
**Why it is relevant:**
No-Show Rate is a key indicator because the primary objective of the analysis is to understand appointment non-attendance. Monitoring this KPI can help HealthConnect understand the scale of missed appointments and establish a baseline for future improvement efforts.

## KPI 2 — Attendance Rate
**Linked Business Question:** BQ1 — What proportion of scheduled appointments are attended, missed or cancelled?
**Definition:**
The percentage of scheduled appointments that result in an attended appointment.
**Why it is relevant:**
Attendance Rate provides a direct measure of successful appointment attendance. It complements the No-Show Rate and provides a broader view of how effectively scheduled appointments are being attended

## KPI 3 — Cancellation Rate
**Linked Business Question:** BQ1 — What proportion of scheduled appointments are attended, missed or cancelled?
**Definition:**
The percentage of scheduled appointments that are recorded as cancelled.
**Why it is relevant:**
Cancellation Rate is important because cancellations represent a different appointment outcome from no-shows. Tracking this KPI separately prevents cancellations and missed appointments from being treated as the same behaviour.

## KPI 4 — Reminder Coverage Rate
**Linked Business Question:** BQ2 — Is reminder activity associated with appointment attendance and no-show patterns?
**Definition:**
The percentage of scheduled appointments for which a reminder was sent.
**Why it is relevant:**
This KPI measures the extent to which the reminder process is being applied across scheduled appointments. It can support subsequent investigation into whether reminder activity is associated with appointment outcomes

## KPI 5 — Repeat No-Show Rate
**Linked Business Question:** BQ3 — Are patients with a history of previous no-shows more likely to miss future appointments?
**Definition:**
The proportion of appointments resulting in a no-show among patients with a history of previous no-shows.
**Why it is relevant:**
This KPI can help assess whether previous no-show behaviour is associated with future appointment non-attendance and whether repeat no-show behaviour may represent an important pattern for further investigation.

## 5.1 KPI Summary
|KPI Linked Business Question Purpose|
|**No-Show Rat**|e BQ1 Measures the proportion of scheduled appointments that are missed|
|**Attendance Rate**| BQ1 Measures the proportion of scheduled appointments that are attended|
|**Cancellation Rate**| BQ1 Measures the proportion of scheduled appointments that are cancelled|
|**Reminder Coverage Rate**| BQ2 Measures the proportion of appointments for which a reminder was sent|
|**Repeat No-Show Rate**| BQ3 Measures no-show behaviour among patients with previous no-shows|

## 5.2 KPI Selection Summary
Five potential KPIs were selected because they directly support the identified business questions.
No-Show Rate, Attendance Rate and Cancellation Rate provide measures of the main appointment outcomes.
Reminder Coverage Rate supports investigation of the relationship between reminder activity and appointment outcomes.
Repeat No-Show Rate supports investigation of whether previous no-show behaviour is associated with future non-attendance.
These KPIs will be calculated and explored in subsequent stages of the project.

## 6. Initial Analysis Approach

The initial analysis approach will follow a structured process that moves
from data validation and preparation to exploratory analysis, KPI analysis,
visualisation and business interpretation.

SQL and Python will be used primarily for data inspection, validation,
preparation and exploratory analysis, while Power BI will be used in
subsequent stages to communicate the findings through interactive
visualisations.

## 6.1 Data Validation
The first stage will involve validating the dataset identified during the Week 4 data-quality assessment.
SQL will be used to perform structured checks such as record counts, duplicate identification, missing-value assessment and category validation.
Python will be used to independently inspect the dataset structure, data types, missing values, duplicates and relationships between relevant variables.
The Data Dictionary will remain the reference for determining whether values are valid and whether identified missing values represent expected conditions or potential data-quality issues.

## 6.2 Data Preparation
Following the initial assessment, the dataset will be prepared for subsequent analysis.
This stage may include correcting data types, handling appropriately identified missing values, checking inconsistent records and creating analysis-ready variables where necessary.
Any transformations will be documented so that the original dataset remains unchanged and the analysis process remains reproducible.

## 6.3 Exploratory Analysis
Exploratory analysis will investigate appointment attendance, no-shows and cancellations across the important variables identified in Task 3.
The analysis will examine relationships between appointment outcomes and factors such as:
-Previous no-show behaviour
-Reminder activity and reminder channel
-Appointment type
-Appointment day
-Appointment time
-Booking lead time
-Distance to the clinic
-Waiting time
-Age and age group
-Gender

Python and SQL will be used to explore patterns and relationships within the data.

## 6.4 KPI Analysis
The five proposed KPIs identified in Task 5 will be calculated during the subsequent analysis stage.
The analysis will use the KPIs to assess overall appointment outcomes, reminder coverage and repeat no-show behaviour.
The KPI results will be compared across relevant patient and appointment characteristics where appropriate.

## 6.5 Power BI Visualisation
Power BI will be used to communicate the key findings from the analysis.
Potential visualisations will include KPI cards, charts showing appointment outcomes and comparisons across relevant variables.
The final visualisations will be selected based on the business questions and analytical findings rather than creating visuals for every available variable.

## 6.6 Business Interpretation
The final stage will interpret the analytical findings in relation to HealthConnect Clinic's objective of understanding appointment attendance and no-show patterns.
The analysis will distinguish between observed associations and causal relationships. Findings will be used to identify important patterns that may warrant further investigation or potential operational consideration.

## 6.7 Initial Analysis Workflow
The planned analysis will follow the sequence below:
Data Dictionary Review ↓ Data Quality Assessment ↓ Data Preparation ↓ Exploratory Data Analysis ↓ KPI Calculation and Analysis ↓ Power BI Visualisation ↓ Business Interpretation ↓ Findings and Recommendations

# 7. Week 4 Conclusion
The Week 4 initial analysis established an understanding of the HealthConnect appointment dataset and its suitability for investigating appointment attendance and no-show patterns.
The Data Dictionary and dataset structure were reviewed, followed by an initial data-quality assessment covering missing values, duplicates, appointment outcomes, historical appointment consistency, reminder data, date fields, booking lead time and numerical variables.
Important variables relevant to appointment attendance and no-shows were identified across appointment outcomes, patient history, reminder activity, appointment scheduling, booking information, operational factors and demographics.
Seven business questions were developed to guide the subsequent analysis. Five potential KPIs were also identified and justified based on their relevance to the business questions.
The proposed analytical approach will use SQL and Python for data validation, preparation and exploratory analysis, followed by Power BI for visualisation and communication of findings.
The next stage of the project will focus on preparing the data, conducting deeper exploratory analysis and calculating the proposed KPIs.